In [5]:
graph = {
    'oradea' : [('zerind', 71), ('sibiu', 151)],
    'zerind' : [('oradea', 71), ('arad', 75)],
    'sibiu' : [('oradea', 151), ('arad', 140), ('fagaras', 99), ('rimnicu vilcea', 80)],
    'arad' : [('zerind', 75), ('sibiu', 140), ('timisoara', 118)],
    'fagaras' : [('sibiu', 99), ('bucharest', 211)],
    'rimnicu vilcea' : [('sibiu', 80), ('pitesti', 97), ('craiova', 146)],
    'timisoara' : [('arad', 118), ('lugoj', 111)],
    'lugoj' : [('timisoara', 111), ('mehadia', 70)],
    'mehadia' : [('lugoj', 70), ('drobeta', 75)],
    'drobeta' : [('mehadia', 75), ('craiova', 120)],
    'craiova' : [('drobeta', 120), ('pitesti', 138), ('rimnicu vilcea', 146)],
    'pitesti' : [('craiova', 138), ('rimnicu vilcea', 97), ('bucharest', 101)],
    'bucharest' : [('giurgiu', 90), ('pitesti', 101), ('fagaras', 211), ('urziceni', 85)],
    'giurgiu' : [('bucharest', 90)],
    'urziceni' : [('bucharest', 85), ('hirsova', 98), ('vaslui', 142)],
    'eforie' : [('hirsova', 86)],
    'hirsova' : [('eforie', 86), ('urziceni', 98), ('vaslui', 142)],
    'vaslui' : [('hirsova', 142), ('lasi', 92)],
    'lasi' : [('vaslui', 92), ('neamt', 87)],
    'neamt' : [('lasi', 87)]
}


In [6]:
hurestic = {
    'oradea' : 380,
    'zerind' : 374,
    'sibiu' : 253,
    'arad' : 366,
    'fagaras' : 176,
    'rimnicu vilcea' : 193,
    'timisoara' : 329,
    'lugoj' : 244,
    'mehadia' : 241,
    'drobeta' : 242,
    'craiova' : 160,
    'pitesti' :  100,
    'bucharest' : 0,
    'giurgiu' : 77,
    'urziceni' :  80,
    'eforie' : 161,
    'hirsova' : 151,
    'vaslui' : 199,
    'lasi' : 226,
    'neamt' :234
}



In [9]:
import heapq

def hill_climb(start, goal, graph, heuristic):

    try:

        if start not in graph:
            raise KeyError

        if goal not in graph:
            raise KeyError

        if start == goal:
            print("\nAlready on Goal")
            return [start]

        priority_queue = []
        total_cost = 0

        heapq.heappush(priority_queue, (heuristic[start], {'current': start}))

        while priority_queue:

            heuristic_value, city = heapq.heappop(priority_queue)

            current = city['current']

            print(f"\nCurrent: {current}, Heuristic: {heuristic_value}")

            if current == goal:
                print("\nGoal Reached")
                print("Total Cost:", total_cost)
                return total_cost

            best = None

            for neighbour, cost in graph[current]:

                if heuristic[neighbour] < heuristic[current]:

                    if best is None or heuristic[neighbour] < heuristic[best[0]]:
                        best = (neighbour, cost)

            if best is None:
                print("\nStuck at local maximum")
                return total_cost

            total_cost += best[1]

            heapq.heappush(
                priority_queue,
                (heuristic[best[0]], {'current': best[0]})
            )

    except KeyError:
        print("not in graph")

start = 'arad'
goal = 'bucharest'
print('\n')

cost = hill_climb(start, goal, graph, hurestic)

print(f"\ncost : {cost}")






Current: arad, Heuristic: 366

Current: sibiu, Heuristic: 253

Current: fagaras, Heuristic: 176

Current: bucharest, Heuristic: 0

Goal Reached
Total Cost: 450

cost : 450


In [17]:
import heapq

def local_beam_search(start, goal, graph, heuristic):

    try:

        if start not in graph:
            raise KeyError

        if goal not in graph:
            raise KeyError

        if start == goal:
            print("\nAlready on Goal")
            return 0

        beam = [(start, 0)]
        path = [start]
        total_cost = 0
        step = 1

        while True:

            print(f"\nStep {step}")

            print("Current Beam:")
            for city, cost in beam:
                print(f"{city}  h={heuristic[city]}")

            for city, cost in beam:
                if city == goal:
                    print("\nGoal Reached")
                    print("Path :", path)
                    print("Total Cost :", total_cost)
                    return total_cost

            best_current = min(heuristic[city] for city, cost in beam)

            best = []

            for city, cost in beam:

                for neighbour, edge_cost in graph[city]:

                    heapq.heappush(
                        best,
                        (heuristic[neighbour], neighbour, city, edge_cost)
                    )

            new_beam = []
            used = set()

            while best and len(new_beam) < 2:

                h, neighbour, parent, edge_cost = heapq.heappop(best)

                if neighbour not in used:
                    used.add(neighbour)
                    new_beam.append((neighbour, edge_cost))

            print("\nNew Beam:")
            for city, cost in new_beam:
                print(f"{city}  h={heuristic[city]}")

            best_new = min(heuristic[city] for city, cost in new_beam)

            if best_new >= best_current:
                print("\n Stuck in Local Minimum / Plateau")
                return total_cost

           
            best_city = new_beam[0]

            path.append(best_city[0])
            total_cost += best_city[1]

            beam = new_beam
            step += 1

    except KeyError:
        print("City not in graph")


start = "arad"
goal = "bucharest"

cost = local_beam_search(start, goal, graph, heuristic)

print("\nCost :", cost)


Step 1
Current Beam:
arad  h=366

New Beam:
sibiu  h=253
timisoara  h=329

Step 2
Current Beam:
sibiu  h=253
timisoara  h=329

New Beam:
fagaras  h=176
rimnicu vilcea  h=193

Step 3
Current Beam:
fagaras  h=176
rimnicu vilcea  h=193

New Beam:
bucharest  h=0
pitesti  h=100

Step 4
Current Beam:
bucharest  h=0
pitesti  h=100

Goal Reached
Path : ['arad', 'sibiu', 'fagaras', 'bucharest']
Total Cost : 450

Cost : 450


In [19]:
def simulated_annealing(start, goal, graph, heuristic):
    try:

        if start not in graph:
            raise KeyError

        if goal not in graph:
            raise KeyError

        current = start
        temperature = 100
        total_cost = 0
        path = [current]


        while temperature >= 1:

            print("\nCurrent City :", current)
            print("Heuristic :", heuristic[current])
            print("Temperature :", round(temperature, 2))

            if current == goal:
                print("\nGoal Reached!")
                break

            next_city, distance = neighbors[0]

            delta_e = heuristic[next_city] - heuristic[current]

            print("Selected Neighbor :", next_city)

            if delta_e < 0:
                current = next_city
                total_cost += distance
                path.append(current)

            else:
                probability = math.exp(-delta_e / temperature)
                print("Delta e =", delta_e)
                print("Acceptance Probability =", round(probability, 4))

                r = random.random()
                print("Random Number =", round(r, 4))

                
            temperature *= 0.8

        if temperature < 1 and current != goal:
            print("\nSearch Terminated ")

        print("\nVisited Path")
        print(path)

        print("Total Path Cost =", total_cost)

    except KeyError:
        print("not in graph")





simulated_annealing("Arad", "Bucharest", graph, heuristic)

not in graph
